In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from typing import Iterable
from typing import Dict, Tuple
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

In [2]:
DATA_DIR  = "../../flatten/processed-data"
DATA_FILE = f"{DATA_DIR}/20251023-s288c-annotated-PPI-net.pkl"
nodes     = pd.read_pickle(DATA_FILE)

In [3]:
# determine threshold for hubs based on quantile cutoff for degree centrality
cut          = 0.10
threshold    = nodes["degree_centrality"].quantile(1 - cut)

# add a boolean hub column: True if degree_centrality >= threshold, else False
nodes["hub"] = nodes["degree_centrality"] >= threshold
n_hubs       = nodes["hub"].sum()
n_nothubs    = (~nodes["hub"]).sum()

print(f"Threshold for defining a hub in MS data is {threshold:.5f}; there are {n_hubs} hubs and {n_nothubs} non hubs")

# filter to nodes with a UniProtKB-AC value, meaning their UniProt annotations could be extracted
nodes        = nodes[~nodes["signalP_trimmed_sequence"].isna()]

Threshold for defining a hub in MS data is 0.01182; there are 393 hubs and 3534 non hubs


In [4]:
nodes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3876 entries, 0 to 3926
Columns: 201 entries, node to hub
dtypes: bool(3), float64(94), int64(40), object(64)
memory usage: 5.9+ MB


In [5]:
def analyze_disorder_vs_hubs(nodes: pd.DataFrame) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    For each is_disordered_* column, build a 2x2 contingency table:

                 disordered   not_disordered
        hub           a              b
        not_hub       c              d

    and compute Fisher's exact test (two-sided) with table [[a, b], [c, d]].

    Returns
    -------
    tables : dict
        {column_name: 2x2 DataFrame contingency table}
    summary : pd.DataFrame
        One row per disorder column with counts, odds_ratio, p_value.
    """

    # auto-detect all disorder columns
    disorder_cols = [c for c in nodes.columns if c.startswith("is_disordered_")]

    tables: Dict[str, pd.DataFrame] = {}
    summary_rows = []

    for col in disorder_cols:
        # 2x2 crosstab: hub vs is_disordered_X
        ct = pd.crosstab(nodes["hub"], nodes[col])

        # ensure full 2x2 with consistent ordering and names
        ct = ct.reindex(index=[False, True], columns=[False, True], fill_value=0)
        ct.index = ["not_hub", "hub"]
        ct.columns = ["not_disordered", "disordered"]

        tables[col] = ct

        # extract counts
        a = int(ct.loc["hub", "disordered"])
        b = int(ct.loc["hub", "not_disordered"])
        c = int(ct.loc["not_hub", "disordered"])
        d = int(ct.loc["not_hub", "not_disordered"])

        # Fisher's exact test (two-sided)
        # table layout: [[hub & disordered, hub & not_disordered],
        #                [not_hub & disordered, not_hub & not_disordered]]
        odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative="two-sided")

        summary_rows.append(
            {
                "column": col,
                "hub_disordered": a,
                "hub_not_disordered": b,
                "not_hub_disordered": c,
                "not_hub_not_disordered": d,
                "odds_ratio": odds_ratio,
                "p_value": p_value,
            }
        )

    summary = pd.DataFrame(summary_rows)
    return tables, summary

In [6]:
tables, summary = analyze_disorder_vs_hubs(nodes)

# look at one contingency table
print(tables["is_disordered_0.50"])

# look at stats across thresholds, sorted by p-value
print(summary.sort_values("p_value").head())

         not_disordered  disordered
not_hub            1779        1712
hub                 196         189
                column  hub_disordered  hub_not_disordered  \
15  is_disordered_0.80              43                 342   
16  is_disordered_0.85              29                 356   
17  is_disordered_0.90              20                 365   
6   is_disordered_0.35             258                 127   
14  is_disordered_0.75              86                 299   

    not_hub_disordered  not_hub_not_disordered  odds_ratio   p_value  
15                 623                    2868    0.578807  0.000785  
16                 458                    3033    0.539455  0.001150  
17                 285                    3206    0.616390  0.045329  
6                 2163                    1328    1.247262  0.052444  
14                 932                    2559    0.789735  0.067243  


In [ ]:
tables